# Step 5b - Taxonomy v0.3 applied to the Step 5 pre-annotation

**Purpose.** Test taxonomy v0.3 (`outputs/s4b_taxonomy/taxonomy_v0.3.json`) on the Step 5 pre-annotation of all 1,500 advertisements. Step 5 spans are re-expressed in v0.3 (expression code kept separate from the bias judgment), joined with the section 7 overlap-test result of Step 4b, and counted at span, unit and advertisement level.

**What this is.** A re-expression and test of existing Step 5 output. The lexicon, the rules, the Step 5 files and the v0.3 definitions are not changed. The six open decisions of Step 4b are not applied; where they matter, the affected spans are listed, not altered.

**What this is not.** No judgment is made (`final_bias_judgment` is `pending` everywhere), no human review is involved, and counts describe what the rules flag, not how much bias exists in the dataset.

**Inputs.** `outputs/s5_preannotation/` (spans, units, ad context), `outputs/analysis/units.csv`, `outputs/s4b_taxonomy/crosswalk_v02_to_v03.csv`, `outputs/s4b_taxonomy/overlap_test_span_results.csv`, `outputs/s4b_taxonomy/taxonomy_v0.3.json`.

**Outputs** (`outputs/s5b_taxonomy_test/`): `preannotation_spans_v0.3.csv`, `preannotation_units_v0.3.csv`, `ad_context_v0.3.csv`, the summary tables and `validation_checks.csv`.

In [1]:
from pathlib import Path
import json
import re

import pandas as pd

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 220)

OUT = Path("outputs") / "s5b_taxonomy_test"
OUT.mkdir(parents=True, exist_ok=True)

S5 = Path("outputs") / "s5_preannotation"
S4B = Path("outputs") / "s4b_taxonomy"
rd = lambda p: pd.read_csv(p, encoding="utf-8-sig")

units_all = rd("outputs/analysis/units.csv")
s5_spans = rd(S5 / "preannotation_spans.csv")
s5_units = rd(S5 / "preannotation_units.csv")
s5_ads = rd(S5 / "ad_context.csv")
cw = rd(S4B / "crosswalk_v02_to_v03.csv")
chk = rd(S4B / "overlap_test_span_results.csv")
tax = {t["code"]: t for t in json.load(open(S4B / "taxonomy_v0.3.json", encoding="utf-8"))["labels"]}
print(f"{units_all['record_id'].nunique():,} ads, {len(units_all):,} units; Step 5 spans {len(s5_spans):,}; v0.3 crosswalk {len(cw):,}; taxonomy codes {sorted(tax)}")

1,500 ads, 7,971 units; Step 5 spans 1,218; v0.3 crosswalk 1,217; taxonomy codes ['E1', 'E2', 'E3', 'G1', 'I1', 'I2', 'INC', 'P', 'U']


## 1. Re-express each Step 5 span in v0.3

Each Step 5 span is matched to its v0.3 crosswalk row on unit, span offsets and span text. The match must be one-to-one. The one span rejected by the Step 5 rule (no gender word) has no v0.3 row and is kept in a separate table, not dropped silently.

In [2]:
key = ["unit_id", "span_start", "span_end", "span_text"]
print("duplicate keys - Step 5:", int(s5_spans.duplicated(key).sum()), "| crosswalk:", int(cw.duplicated(key).sum()))

assert (cw["unit_id"].values == chk["unit_id"].values).all() and (cw["span_text"].values == chk["span_text"].values).all()     and (cw["expression_code"].values == chk["expression_code"].values).all(), "crosswalk and overlap-test rows are not aligned"
cw = cw.assign(conditions_predict=chk["pred_str"].values, conditions_status=chk["pred_status"].values)

v3cols = key + ["expression_code", "expression_category", "primary_classification", "bias_judgment_prior",
                "final_bias_judgment", "review_status", "u_reason", "notes", "conditions_predict", "conditions_status"]
m = s5_spans.merge(cw[v3cols].rename(columns={"review_status": "v03_review_status", "notes": "v03_notes"}),
                   on=key, how="left", indicator=True)
unmatched = m[m["_merge"] == "left_only"]
print(f"Step 5 spans: {len(s5_spans):,}; matched to v0.3: {(m['_merge'] == 'both').sum():,}; unmatched: {len(unmatched)}")
display(unmatched[["unit_id", "span_text", "v02_candidate", "review_status"]])

spans = m[m["_merge"] == "both"].drop(columns="_merge").copy()
spans["assigned_in_conditions"] = [a in str(p).split("+") for a, p in zip(spans["expression_code"], spans["conditions_predict"])]
spans["taxonomy_check"] = "pass"
spans.loc[spans["conditions_status"].eq("no fit"), "taxonomy_check"] = "no_code_fits"
spans.loc[spans["conditions_status"].eq("overlap"), "taxonomy_check"] = "overlap"
spans.loc[spans["conditions_status"].eq("single") & ~spans["assigned_in_conditions"], "taxonomy_check"] = "conditions_point_elsewhere"
print("rows after joining:", len(spans), "(must equal matched count)")
display(pd.crosstab(spans["v02_candidate"], spans["expression_code"], margins=True))

duplicate keys - Step 5: 0 | crosswalk: 0
Step 5 spans: 1,218; matched to v0.3: 1,217; unmatched: 1


,unit_id,span_text,v02_candidate,review_status
1120,row-1416-s005,preferably MS,none,rejected_by_rule


rows after joining: 1217 (must equal matched count)


expression_code,E1,E2,E3,G1,I1,I2,P,U,All
v02_candidate,,,,,,,,,
E1,317,0,0,0,0,0,0,0,317
E2,0,40,0,0,0,0,0,0,40
E4_or_E5,0,0,5,0,0,0,0,0,5
I1,0,0,0,0,350,0,0,0,350
I2,0,0,0,0,0,379,0,0,379
INC,0,0,0,99,0,0,0,0,99
P,0,0,0,0,0,0,11,0,11
needs_review,0,0,0,0,0,0,0,16,16
All,317,40,5,99,350,379,11,16,1217


## 2. Class distribution under v0.3

One row per code: role, route to the primary classification, and the number of spans, units, advertisements and flagged tokens. Tokens are counted with the same rule as Step 5a (`\w+|[^\w\s]`).

In [3]:
TOK = re.compile(r"\w+|[^\w\s]")
spans["span_tokens"] = spans["span_text"].map(lambda s: len(TOK.findall(str(s))))
role = {"E1": "primary", "E2": "primary", "E3": "primary", "I1": "primary", "I2": "primary",
        "G1": "auxiliary", "INC": "auxiliary", "P": "auxiliary", "U": "review"}
rows = []
for c in ["E1", "E2", "E3", "I1", "I2", "G1", "INC", "P", "U"]:
    s = spans[spans["expression_code"] == c]
    rows.append({"code": c,     "category": tax[c]["category"], "role": role[c],
                     "route": tax[c]["primary_classification"],
                 "spans": len(s), "units": s["unit_id"].nunique(), "ads": s["record_id"].nunique(),
                 "span_tokens": int(s["span_tokens"].sum())})
dist = pd.DataFrame(rows)
display(dist)
dist.to_csv(OUT / "class_distribution_v0.3.csv", index=False, encoding="utf-8-sig")

n_tokens = int(units_all["text_masked"].map(lambda s: len(TOK.findall(str(s)))).sum())
flag_tok = spans.drop_duplicates(["unit_id", "span_start", "span_end"])["span_tokens"].sum()
corpus = pd.DataFrame([
    ("Instances (units)", len(units_all)), ("Source advertisements", units_all["record_id"].nunique()),
    ("Tokens", n_tokens), ("Flagged spans", len(spans)), ("Flagged tokens", int(flag_tok)),
    ("Units with a flagged span", spans["unit_id"].nunique()), ("Ads with a flagged span", spans["record_id"].nunique()),
], columns=["measure", "value"])
display(corpus)
corpus.to_csv(OUT / "corpus_summary_v0.3.csv", index=False, encoding="utf-8-sig")

,code,category,role,route,spans,units,ads,span_tokens
0,E1,Gender restriction or preference,primary,explicit,317,312,311,347
1,E2,Gender-marked occupational wording,primary,explicit,40,40,32,40
2,E3,Generic gendered pronoun,primary,explicit,5,3,2,5
3,I1,Masculine-coded expression,primary,implicit,350,324,240,360
4,I2,Feminine-coded expression,primary,implicit,379,324,275,379
5,G1,Non-restrictive gender listing,auxiliary,none,99,99,98,297
6,INC,Inclusive expression,auxiliary,none,0,0,0,0
7,P,Polysemantic / contextual near miss,auxiliary,none,11,11,11,11
8,U,Unclear expression,review,unclear,16,16,16,16


,measure,value
0,Instances (units),7971
1,Source advertisements,1500
2,Tokens,58064
3,Flagged spans,1217
4,Flagged tokens,1455
5,Units with a flagged span,1073
6,Ads with a flagged span,765


## 3. Route to the primary classification

Span level, unit level and advertisement level. An advertisement takes the highest route among its spans, in the order explicit, implicit, unclear, none. The order is a reporting convention used here, not a rule of the taxonomy.

In [4]:
order = ["explicit", "implicit", "unclear", "none"]
span_route = spans["primary_classification"].value_counts().reindex(order).fillna(0).astype(int)
unit_route = spans.groupby("unit_id")["primary_classification"].agg(lambda s: next(r for r in order if r in set(s)))
ad_route = spans.groupby("record_id")["primary_classification"].agg(lambda s: next(r for r in order if r in set(s)))
route = pd.DataFrame({"spans": span_route, "units (highest route)": unit_route.value_counts().reindex(order).fillna(0).astype(int),
                      "ads (highest route)": ad_route.value_counts().reindex(order).fillna(0).astype(int)})
n_ads = units_all["record_id"].nunique()
route.loc["no span"] = [0, len(units_all) - len(unit_route), n_ads - len(ad_route)]
display(route)
route.to_csv(OUT / "route_counts_v0.3.csv", encoding="utf-8-sig")

both = set(spans.loc[spans["primary_classification"].eq("explicit"), "record_id"]) & set(spans.loc[spans["primary_classification"].eq("implicit"), "record_id"])
print(f"Ads holding both an explicit and an implicit span: {len(both)}")

,spans,units (highest route),ads (highest route)
primary_classification,,,
explicit,362,355,340
implicit,729,593,344
unclear,16,16,9
none,110,109,72
no span,0,6898,735


Ads holding both an explicit and an implicit span: 69


## 4. Taxonomy check on the Step 5 spans

Every span is checked against the operational conditions written in Step 4b (section 7). `pass` means the assigned code is the only code whose conditions hold. The other values are findings for the owner's review; nothing is relabelled.

In [5]:
tc = pd.crosstab(spans["expression_code"], spans["taxonomy_check"], margins=True)
display(tc)
tc.to_csv(OUT / "taxonomy_check_by_code.csv", encoding="utf-8-sig")

flagged = spans[spans["taxonomy_check"] != "pass"]
detail = (flagged.groupby(["taxonomy_check", "expression_code", "conditions_predict"])
          .agg(spans=("unit_id", "size"), units=("unit_id", "nunique"), ads=("record_id", "nunique"),
               example=("span_text", "first")).reset_index())
display(detail)
detail.to_csv(OUT / "taxonomy_check_detail.csv", index=False, encoding="utf-8-sig")
flagged[["unit_id", "record_id", "component", "expression_code", "span_text", "conditions_predict", "taxonomy_check", "text_masked"]] \
    .to_csv(OUT / "taxonomy_check_spans.csv", index=False, encoding="utf-8-sig")
print(f"{len(flagged)} of {len(spans)} spans ({len(flagged)/len(spans):.1%}) are not a clean pass; "
      f"{flagged['record_id'].nunique()} ads contain at least one.")

taxonomy_check,conditions_point_elsewhere,no_code_fits,overlap,pass,All
expression_code,,,,,
E1,6,4,0,307,317
E2,0,1,0,39,40
E3,4,0,0,1,5
G1,0,0,0,99,99
I1,0,0,66,284,350
I2,0,0,43,336,379
P,0,0,0,11,11
U,15,1,0,0,16
All,25,6,109,1077,1217


,taxonomy_check,expression_code,conditions_predict,spans,units,ads,example
0,conditions_point_elsewhere,E1,G1,6,3,3,Male
1,conditions_point_elsewhere,E3,G1,4,2,1,his
2,conditions_point_elsewhere,U,E1,13,13,13,Male
3,conditions_point_elsewhere,U,P,2,2,2,MAN
4,overlap,I1,E2+I1,3,3,3,Leadman
5,overlap,I1,I1+P,63,63,54,Analyst
6,overlap,I2,I2+P,43,40,34,Support


140 of 1217 spans (11.5%) are not a clean pass; 111 ads contain at least one.

## 5. What changed from Step 5 (v0.2) to Step 5b (v0.3)

Span level: how each v0.2 candidate maps to a v0.3 code. Advertisement level: whether the highest route of an advertisement differs between the Step 5 family (explicit, explicit pending context = explicit; pending review = unclear; not bias = none) and the v0.3 route.

In [6]:
chg = pd.crosstab(spans["v02_candidate"], spans["expression_code"])
display(chg)
fam_to_route = {"explicit": "explicit", "explicit (pending context)": "explicit", "implicit": "implicit",
                "pending review": "unclear", "not bias": "none"}
spans["v02_route"] = spans["v02_family"].map(fam_to_route)
assert spans["v02_route"].notna().all()
print("v02_family values:", sorted(spans["v02_family"].dropna().unique()))
v2_ad = spans.groupby("record_id")["v02_route"].agg(lambda s: next(r for r in order if r in set(s)))
cmp = pd.DataFrame({"v02": v2_ad, "v03": ad_route})
ad_changes = pd.crosstab(cmp["v02"], cmp["v03"], margins=True)
display(ad_changes)
ad_changes.to_csv(OUT / "ad_route_v02_vs_v03.csv", encoding="utf-8-sig")

expression_code,E1,E2,E3,G1,I1,I2,P,U
v02_candidate,,,,,,,,
E1,317,0,0,0,0,0,0,0
E2,0,40,0,0,0,0,0,0
E4_or_E5,0,0,5,0,0,0,0,0
I1,0,0,0,0,350,0,0,0
I2,0,0,0,0,0,379,0,0
INC,0,0,0,99,0,0,0,0
P,0,0,0,0,0,0,11,0
needs_review,0,0,0,0,0,0,0,16


v02_family values: ['explicit', 'explicit (pending context)', 'implicit', 'not bias', 'pending review']


v03,explicit,implicit,none,unclear,All
v02,,,,,
explicit,340,0,0,0,340
implicit,0,344,0,0,344
none,0,0,72,0,72
unclear,0,0,0,9,9
All,340,344,72,9,765


## 6. Packaged outputs with the full advertisement

Spans, units and advertisements are written in v0.3. The Step 5 files are left untouched. Each advertisement row carries the masked text with component tags, its v0.3 codes and route, and the taxonomy-check count.

In [7]:
sp_out = spans[["unit_id", "record_id", "employer_id", "component", "text_masked", "span_text", "span_start", "span_end",
                "span_offset_valid", "raw_start", "raw_end", "v02_candidate", "expression_code", "expression_category",
                "primary_classification", "bias_judgment_prior", "final_bias_judgment", "v03_review_status",
                "review_flag", "u_reason", "v03_notes", "conditions_predict", "conditions_status", "taxonomy_check"]]
sp_out.to_csv(OUT / "preannotation_spans_v0.3.csv", index=False, encoding="utf-8-sig")

by_unit = spans.groupby("unit_id")
u = s5_units.drop(columns=["candidate_codes", "n_candidate_spans", "unit_status"]).copy()
u["n_spans_v03"] = u["unit_id"].map(by_unit.size()).fillna(0).astype(int)
u["codes_v03"] = u["unit_id"].map(by_unit["expression_code"].agg(lambda s: "|".join(sorted(set(s))))).fillna("")
u["route_v03"] = u["unit_id"].map(unit_route).fillna("")
u["taxonomy_check"] = u["unit_id"].map(by_unit["taxonomy_check"].agg(lambda s: "pass" if set(s) == {"pass"} else "|".join(sorted(set(s) - {"pass"})))).fillna("")
u.to_csv(OUT / "preannotation_units_v0.3.csv", index=False, encoding="utf-8-sig")

by_ad = spans.groupby("record_id")
a = s5_ads[["record_id", "n_units", "ad_text_masked", "unit_notes", "guideline"]].copy()
a["n_spans_v03"] = a["record_id"].map(by_ad.size()).fillna(0).astype(int)
a["codes_v03"] = a["record_id"].map(by_ad["expression_code"].agg(lambda s: "|".join(sorted(set(s))))).fillna("")
a["route_v03"] = a["record_id"].map(ad_route).fillna("")
a["n_taxonomy_check_not_pass"] = a["record_id"].map(by_ad["taxonomy_check"].agg(lambda s: int((s != "pass").sum()))).fillna(0).astype(int)
a["taxonomy"] = "taxonomy_v0.3.json"
a.to_csv(OUT / "ad_context_v0.3.csv", index=False, encoding="utf-8-sig")
print(len(sp_out), "spans;", len(u), "units;", len(a), "ads written to", OUT)

1217 spans; 7971 units; 1500 ads written to outputs\s5b_taxonomy_test


### Examples with full context

The first advertisement (by id) for each of: a clean explicit span, a G1 listing, a span the taxonomy check did not pass, and a U span. Shown as stored, masked.

In [8]:
def show(rid):
    r = a[a["record_id"] == rid].iloc[0]
    print(f"=== {rid} | route {r.route_v03} | codes {r.codes_v03} | taxonomy-check not pass: {r.n_taxonomy_check_not_pass}")
    print(r.ad_text_masked[:1200])
    for x in spans[spans["record_id"] == rid].itertuples():
        print(f"   -> {x.expression_code:<3} {x.primary_classification:<9} '{x.span_text}' [{x.taxonomy_check}]")
    print()

picks = []
for label, q in [("explicit pass", (spans["expression_code"] == "E1") & (spans["taxonomy_check"] == "pass")),
                 ("G1", spans["expression_code"] == "G1"),
                 ("not pass", spans["taxonomy_check"] != "pass"),
                 ("U", spans["expression_code"] == "U")]:
    picks.append(sorted(spans.loc[q, "record_id"])[0])
for rid in dict.fromkeys(picks):
    show(rid)

=== row-0022 | route explicit | codes E1 | taxonomy-check not pass: 0
[job_title] Visual Artist
[qualification] Male
[qualification] Graduate of any four (4) year Computer or Drafting related courses
[qualification] With skills in designing
   -> E1  explicit  'Male' [pass]

=== row-0003 | route none | codes G1 | taxonomy-check not pass: 0
[job_title] Human Resource Assistant
[qualification] Female/Male
[qualification] Graduate of BS Psychology, BSBA Human Resource Management, BS ~Behavioral Science or other related courses
[qualification] With at least 1 year experience in human resources functions
[qualification] With knowledge in implementing all human resources faucet
   -> G1  none      'Female/Male' [pass]

=== row-0012 | route implicit | codes I1|I2|U | taxonomy-check not pass: 1
[job_title] Stock Clerk
[body_other] Male
[body_other] 20 Years old Above
[body_other] At least High School Graduate or College Level with experience as Stock Clerk or Inventory Staff
[body_other] Fresh

## 7. Validation checks

In [9]:
checks = []
def rec(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})

rec("Step 5 spans = matched + unmatched", len(s5_spans) == len(spans) + len(unmatched), f"{len(s5_spans)} = {len(spans)} + {len(unmatched)}")
rec("only the rule-rejected span is unmatched", set(unmatched["review_status"]) <= {"rejected_by_rule"} and len(unmatched) == 1)
rec("no duplicate span keys after joins", not spans.duplicated(key + ["expression_code"]).any())
rec("every span has a v0.3 code that exists in the taxonomy", spans["expression_code"].isin(list(tax)).all())
rec("every span route is explicit / implicit / none / unclear", spans["primary_classification"].isin(order).all())
rec("final_bias_judgment is pending everywhere", spans["final_bias_judgment"].eq("pending").all())
rec("spans per code sum to total", int(dist["spans"].sum()) == len(spans))
rec("span routes sum to total", int(span_route.sum()) == len(spans))
rec("ads in route table = all advertisements", int(route["ads (highest route)"].sum()) == n_ads)
rec("units in route table = all units", int(route["units (highest route)"].sum()) == len(units_all))
rec("taxonomy-check values sum to total", int(tc.loc["All", "All"]) == len(spans))
rec("every span lies inside its unit text", all(0 <= s < e <= len(str(t)) for s, e, t in zip(spans.loc[spans["span_offset_valid"].astype(bool), "span_start"].astype(int), spans.loc[spans["span_offset_valid"].astype(bool), "span_end"].astype(int), spans.loc[spans["span_offset_valid"].astype(bool), "text_masked"])))
rec("every unit_id in the outputs exists in units.csv", set(sp_out["unit_id"]) <= set(units_all["unit_id"]))
rec("Step 5 output files unchanged in row count", len(s5_spans) == 1218 and len(s5_units) == 7971 and len(s5_ads) == 1500)
rec("class distribution matches the Step 4b crosswalk (excluding the rejected span)", dist.set_index("code")["spans"].to_dict() == cw["expression_code"].value_counts().reindex(dist["code"]).fillna(0).astype(int).to_dict())
checks = pd.DataFrame(checks)
checks.to_csv(OUT / "validation_checks.csv", index=False, encoding="utf-8-sig")
display(checks)
print(f"{int(checks['passed'].sum())} of {len(checks)} checks passed.")

,check,passed,detail
0,Step 5 spans = matched + unmatched,True,1218 = 1217 + 1
1,only the rule-rejected span is unmatched,True,
2,no duplicate span keys after joins,True,
3,every span has a v0.3 code that exists in the taxonomy,True,
4,every span route is explicit / implicit / none / unclear,True,
5,final_bias_judgment is pending everywhere,True,
6,spans per code sum to total,True,
7,span routes sum to total,True,
8,ads in route table = all advertisements,True,
9,units in route table = all units,True,


15 of 15 checks passed.
